# Assignment 3 · CNNs and Transformers
**CSCI3230 / ESTR3108 · Individual programming assignment · 100 points**

**Deadline: 24 November 2026, 23:59 (Hong Kong time, UTC+8).**

If you have any questions, please contact the TA responsible for this assignment: **Jihai Zhang** ([jihaizhang@link.cuhk.edu.hk](mailto:jihaizhang@link.cuhk.edu.hk)).

Read and acknowledge the AI-use and academic-integrity requirements at the beginning of each part. Expected active work: approximately 5–7 hours, plus training time.

You will design a CNN that classifies clothing images while remaining unchanged under circular translations, and complete a small causal Transformer that autoregressively generates ordered answers to multiple queries using a dictionary in its current context.

This notebook is the specification and runnable guide. Your submitted `.py` files and weights are evaluated by a fresh Python process. A running notebook kernel is not part of your submission.

| Part | What is graded | Points |
|---|---|---:|
| CNN | Loading, shapes, deterministic and batch-independent inference | 5 |
| CNN | Translation-invariance checks, including fresh initialization | 20 |
| CNN | Accuracy on translated Fashion-MNIST images | 25 |
| Transformer | Attention outputs, weights, and gradients | 12 |
| Transformer | Causal mask and next-token alignment | 8 |
| Transformer | Valid output-format specification | 5 |
| Transformer | Generation accuracy using your implementation | 20 |
| Transformer | One-page PDF analysis (submission credit, marked by the TA) | 5 |

This is one complete assignment package. Implement three Transformer functions and design a JSON output format. The grading server plugs your functions into the fixed course model, trains it and scores its generated answers; you submit no Transformer weights or training records. Submit a one-page PDF analysis as described in Section 2.7. The automated components total 95 points; the TA awards the remaining 5 points for submitting the PDF.

## Setup and first-run check

Unzip the student release and open this notebook in its `assignment3` directory. Install compatible PyTorch and torchvision versions, plus the packages in `requirements.txt`.

```bash
python -m pip install -r requirements.txt
```

The reference environment uses Python 3.12, PyTorch 2.13 and torchvision 0.28. Interface checks were also tested in the local Python 3.9 / PyTorch 2.0 environment. Formal scoring runs on **CPU with two PyTorch threads**, in the TA's fixed environment.

CNN training defaults to CPU. Set `DEVICE` explicitly if you have a working CUDA or MPS environment for the CNN. Format experiments always use the fixed CPU/two-thread policy. MPS was tested for CNN training in the course environment; older PyTorch MPS installations can behave differently. Running this notebook as supplied does not train or download a dataset.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import numpy as np
import matplotlib.pyplot as plt
import torch

ROOT = Path.cwd()
assert (ROOT / "check_submission.py").exists(), "Open the notebook from the assignment3 directory."
DATA_DIR = ROOT / "data"
SUBMISSION_DIR = ROOT / "submission"
DEVICE = "cpu"
print("Python:", sys.version.split()[0], "| PyTorch:", torch.__version__)
print("Assignment files found in the current directory.")

### Files you edit and files you run

```text
Assignment3.ipynb              ← read the task and run examples
submission/cnn.py             ← design your CNN
submission/transformer.py     ← complete three TODO tensor functions
submission/config.json        ← CNN settings may change; Transformer settings are fixed
train_cnn.py                  ← provided training and checkpoint selection
submission/format.json        ← choose the output record template
submission/analysis.pdf       ← write the one-page format comparison
submission/acknowledgement.json ← complete both required acknowledgements
train_transformer.py          ← fixed-budget format experiment; optional baseline comparison
check_submission.py           ← public checks, same evaluator as formal grading
pack_submission.py            ← validate interfaces and create your submission ZIP
assignment_lib/               ← course-owned code; do not modify or submit replacements
example_submission/           ← runnable, intentionally incorrect example + random weights
```

The example submission demonstrates the **loading contract**, not a solution. It passes interface checks and deliberately fails several correctness checks. Do not submit it as your answer.

In [ ]:
example_check = subprocess.run(
    [sys.executable, "check_submission.py", "--submission", "example_submission",
     "--interfaces-only", "--result", "example_interface_result.json"],
    capture_output=True, text=True)
print(example_check.stdout.replace(str(ROOT), "."))
if example_check.returncode:
    print(example_check.stderr)
    raise RuntimeError("Fix the environment before working on the models.")

### General execution requirements

Use only the supplied dependencies. Imports must have no training, download, prompting, or file-writing side effects. Use the input/model device; do not hard-code `cuda` or `mps`. Evaluation must be deterministic in `model.eval()`, and predictions must not depend on other examples in the batch.

Each CNN weights file is limited to **8 MB**; expanded submission files together are limited to **12 MB**. The submitted files and final packaging instructions are listed under **Check and submit** at the end of this notebook.

## Part 1 · CNN — 50 points

### Acknowledgement

You may discuss this assignment with AI tools and use AI-assisted programming. However, delegating the entire solution to an autonomous agent without reviewing and understanding its work is prohibited. You are responsible for understanding every part of your submitted solution.

After the deadline, randomly selected students will attend a one-to-one code review with a TA. You must clearly explain the principles of your solution and reproduce your results live during the session. AI assistance is permitted during the code review.

Plagiarism and attempts to attack, tamper with, hack, or circumvent the grading system are prohibited. Violations may result in zero marks for the entire assignment or more serious consequences.

**Required acknowledgement:** in `submission/acknowledgement.json`, replace the empty value of the **`cnn`** field with the exact lowercase string **`"yes"`** to confirm that you have read, understood, and agree to follow these requirements. Keep both fields in the same file. The supplied template is:

```json
{
  "cnn": "",
  "transformer": ""
}
```

**Both fields must be `"yes"` in your submitted file. If the file is missing, is not valid JSON, or either field is missing or not `"yes"`, the entire assignment receives 0/100, including the PDF component.** Packaging and public checks do not validate this acknowledgement or remind you to complete it. It is your responsibility to follow these instructions; formal grading enforces this requirement.

### 1.1 · Task and circular translation

Fashion-MNIST has ten classes, in this output-index order:

`0 T-shirt/top, 1 Trouser, 2 Pullover, 3 Dress, 4 Coat, 5 Sandal, 6 Shirt, 7 Sneaker, 8 Bag, 9 Ankle boot`.

We use the official 60,000 training images with a fixed split: **50,000 training / 10,000 validation**. The helper in `assignment_lib.data` defines the split. Formal accuracy is measured on the official 10,000-image test set after translations. Test images/labels must not be used for training, architecture selection, or checkpoint selection. The benchmark is public; calling the grading split “test” does not make its labels secret.

All preprocessing is fixed: `float32 pixels / 127.5 - 1`. Train on the unshifted training images, without augmentation or external/pretrained weights.

#### What does translation mean here?

Use `torch.roll` on the two spatial dimensions. Content leaving one edge wraps around to the opposite edge. Horizontal and vertical shifts are integers in `[-5, 5]`, independently chosen for each image. No pixels are cropped or filled with zeros.

For logits $f(x)$, require
$$
f(T_{\Delta h,\Delta w}x) \approx f(x).
$$

The requirement is about **each logit vector**, not merely equal overall accuracy or the same predicted class. It must hold structurally, including before training. Tests run in evaluation mode and allow elementwise error `atol + rtol * abs(reference)`, with **atol = 2e-5, rtol = 2e-4**.

Predicted class IDs must also stay identical. To prevent tiny output scaling from bypassing the tolerance,
the evaluator additionally centers each logit vector, divides it by its L2 norm (zero vectors stay zero),
and compares these normalized vectors with the same tolerance.

Consider how boundary handling, spatial downsampling, and final aggregation affect this property. The supplied CNN is a working classifier template that does **not** satisfy it.

In [ ]:
with np.load(ROOT / "assignment_lib" / "public_fashion_examples.npz") as examples:
    images = torch.from_numpy(examples["images"].copy())
    labels = examples["labels"].copy()
index = 0
shift = (4, -5)
translated = torch.roll(images[index], shifts=shift, dims=(-2, -1))
fig, axes = plt.subplots(1, 2, figsize=(6, 3))
for ax, image, title in zip(axes, [images[index], translated],
                            ["Original", f"Circular shift {shift}"]):
    ax.imshow(image[0], cmap="gray", vmin=-1, vmax=1)
    ax.set_title(title); ax.axis("off")
fig.tight_layout()
plt.show()
print("Both images have the same class label:", int(labels[index]))

### 1.2 · Model interface and checkpoint — 5 points

`submission/cnn.py` exposes `build_model(config: dict) -> torch.nn.Module`. Construct the CNN on CPU without loading data/checkpoints, training or downloading. The evaluator loads its weights.

| Input | Required output |
|---|---|
| float32 `[B, 1, 28, 28]`, normalized to `[-1, 1]` | floating-point logits `[B, 10]` |

Return logits, not class IDs or probabilities. Support different batch sizes, deterministic evaluation, and batch-independent predictions. These interface checks are worth **5 points**.

Save a flat `state_dict` with tensor values:

```python
torch.save({k: v.detach().cpu() for k, v in model.state_dict().items()}, "cnn_weights.pt")
```

The evaluator uses `torch.load(..., map_location="cpu", weights_only=True)` and `load_state_dict(..., strict=True)`. Do not save the entire Python model or rename state-dictionary keys.

### 1.3 · Design and training limits

- Modify `submission/cnn.py` and the `cnn` object in `config.json`. Use a learned convolutional feature extractor with at least one `nn.Conv2d` and **at most 200,000 total parameters**.
- Use one model and one forward evaluation per input. Ensembles, enumeration/averaging of translated copies, memorized labels, test-specific branches, network access, and reading files inside `forward` are not allowed.
- Keep the supplied training pipeline: AdamW, learning rate `1e-3`, weight decay `1e-4`, batch size 128, seed 3230, **at most 10 epochs per run**. You may compare architectures using the validation split. Each run starts from random initialization.
- The trainer saves the epoch with the best validation accuracy. Do not select checkpoints using the test set.

### 1.4 · Train your CNN

First run downloads Fashion-MNIST into your chosen data directory:

```bash
python train_cnn.py --submission submission --data-dir data --download --device cpu
```

Add `--device cuda` or `--device mps` only for a working backend. Existing weights are protected; `--overwrite` explicitly replaces them. `--smoke` performs two updates on a tiny subset to check the pipeline and is not a complete training run. Never submit smoke-test weights as a trained solution.

In [ ]:
RUN_CNN_TRAINING = False
if RUN_CNN_TRAINING:
    subprocess.run([sys.executable, "train_cnn.py", "--submission", str(SUBMISSION_DIR),
                    "--data-dir", str(DATA_DIR), "--download", "--device", DEVICE], check=True)
else:
    print("CNN training is disabled. Edit cnn.py, then enable this cell or use the command above.")

### 1.5 · Invariance and accuracy scoring — 45 points


The four invariance groups are worth 5 points each. They cover loaded weights and fresh initialization, synthetic inputs and real images, and mixed per-image shifts. A group earns its points when all its checks pass. Hidden tests use the same contract with additional shifts/input samples. Finite tests provide evidence of invariance; they are not a mathematical proof.

The 25 performance points are available only when the CNN interface and **all four** invariance groups pass. Otherwise accuracy is still reported, but the performance component is zero. For translated-test accuracy $a$ expressed as a fraction:

$$
S_{\mathrm{CNN,acc}}=25\;\mathrm{clip}\!\left(\frac{a-0.75}{0.88-0.75},0,1\right).
$$

Thus 75% or below earns zero performance points; 88% or above earns all 25. Scores increase linearly in between. Scores do not depend on other students' rankings. Correctness points remain separately available.

## Part 2 · Transformer — 50 points

### Acknowledgement

You may discuss this assignment with AI tools and use AI-assisted programming. However, delegating the entire solution to an autonomous agent without reviewing and understanding its work is prohibited. You are responsible for understanding every part of your submitted solution.

After the deadline, randomly selected students will attend a one-to-one code review with a TA. You must clearly explain the principles of your solution and reproduce your results live during the session. AI assistance is permitted during the code review.

Plagiarism and attempts to attack, tamper with, hack, or circumvent the grading system are prohibited. Violations may result in zero marks for the entire assignment or more serious consequences.

**Required acknowledgement:** in `submission/acknowledgement.json`, replace the empty value of the **`transformer`** field with the exact lowercase string **`"yes"`** to confirm that you have read, understood, and agree to follow these requirements. Keep both fields in the same file. The supplied template is:

```json
{
  "cnn": "",
  "transformer": ""
}
```

**Both fields must be `"yes"` in your submitted file. If the file is missing, is not valid JSON, or either field is missing or not `"yes"`, the entire assignment receives 0/100, including the PDF component.** Packaging and public checks do not validate this acknowledgement or remind you to complete it. It is your responsibility to follow these instructions; formal grading enforces this requirement.

### 2.1 · Task: answer multiple dictionary queries

You are given a **fixed 74,659-parameter causal Transformer** and **exactly 300 training updates**. Implement its attention, causal mask and target alignment correctly, then improve answer accuracy by designing the output format. Architecture, initialization, input prompt, training data, optimizer, loss rule and decoding policy are fixed.

**Your implementation is used throughout this part.** First implement the Transformer functions, then train a value-only baseline, and finally explore output formats using that same implementation. The final accuracy is measured after training the fixed model with **your functions and your chosen format**. The course supplies the architecture and training loop; your functions remain in the model during both training and generation.

Each example has 2–8 dictionary entries and 2–4 distinct query keys. Keys in a dictionary are unique; values may repeat. Every queried key appears in that example's dictionary, and the same key can map to a different value in another example.

```text
Dictionary: K2=V3, K7=V1, K0=V6
Queries:    K7 K0 K2
Baseline response: V1 V6 V3 <EOS>

Complete baseline training sequence:
<BOS> K2 V3 K7 V1 K0 V6 <QUERY> K7 K0 K2 <SEP> V1 V6 V3 <EOS>

Generation prompt:
<BOS> K2 V3 K7 V1 K0 V6 <QUERY> K7 K0 K2 <SEP>
```

The prompt is identical for every format. Your design changes the response after SEP only. A stable hash of the raw dictionary and ordered queries assigns each problem to train/validation/test **before** the response is encoded. Different formats therefore train on the same underlying problem sequence.

#### Training and generation

During training, one causal forward pass predicts next tokens using teacher forcing. Cross-entropy is averaged over every response token and EOS; prompt and padding targets are excluded. During grading, generation begins with the prompt only and feeds back its own predictions until EOS or a fixed cap. There are no supplied answer tokens, forced keys or vocabulary restrictions.

**Think before editing:** when predicting the next answer, how does a small model know which query it is answering? Could your output representation make useful information explicit? Extra tokens also cost supervision and decoding work; they need not improve accuracy.

**Vocabulary.** Token IDs: PAD/BOS/QUERY/SEP/EOS = 0–4; K0–K15 = 5–20; V0–V7 = 21–28; I1–I4 = 29–32; `|` = 33; `.` = 34.

In [ ]:
from assignment_lib.data import make_queries, format_tokens
from assignment_lib.format_spec import BASELINE, encode_answer
prompts, lengths, targets = make_queries(2, np.random.default_rng(11), response_format=BASELINE)
for prompt, target in zip(prompts, targets):
    print("Prompt:  ", format_tokens(prompt))
    print("Baseline:", format_tokens(target))

### 2.2 · Implement the Transformer functions — 20 points

Complete exactly the three functions below in `submission/transformer.py`. The grader imports them and constructs the fixed course architecture; do not supply your own Transformer model builder. Function tests use fresh weights, and the same functions are used in the actual 300-step training and autoregressive inference. **No Transformer checkpoint is submitted.** Keep the file self-contained using the supplied dependencies.

The course-built model accepts int64 token IDs `[B, T]` and returns logits `[B, T, 35]`. It must support sequence lengths from 1 to 48. The architecture and training budget are specified in Section 2.3.

**A. `scaled_dot_product_attention(q,k,v,allowed_mask=None)` — 12 points**

Q, K, V have shape `[B,H,T,D]`. Return context `[B,H,T,D]` and weights `[B,H,T,T]`:

$$
\operatorname{softmax}(QK^\top/\sqrt{D}+\text{mask bias})V.
$$

The optional mask is bool `[T,T]`, with **True meaning allowed**. Mask before softmax; each tested row has at least one allowed key. Support float32/float64, gradients and the input device. Use basic tensor operations, without built-in attention APIs, NumPy conversion, detach or dropout. Correct context earns 4 points, weights 3, and Q/K/V gradients 5.

**B. `make_causal_mask(sequence_length,device)` — 4 points**

Return bool `[T,T]` on the requested device. Position i may attend to positions j ≤ i, including itself.

**C. `prepare_inputs_and_targets(tokens)` — 4 points**

Complete right-padded int64 `[B,L]` sequences become `(inputs,targets,loss_mask)`, all `[B,L-1]`. Shift targets by one. Select every response-token target and EOS after SEP, excluding prompt and padding targets. This rule must work for any permitted record layout, not just your chosen format. The input position at SEP predicts the first response token.

**Future independence — required for a valid generation experiment**

Changing future input tokens must not change earlier output logits. The evaluator tests your attention and mask inside the fixed, freshly initialized Transformer.

Training uses your functions inside the fixed architecture. The course controls parameter count, initialization, data, optimizer, number of updates and decoding. Before training it checks shapes, gradient connectivity, causality and target alignment; every training batch also checks target/loss-mask alignment. Exceptions are reported with the failing stage. Implement the functions correctly rather than special-casing the tests.

### 2.3 · Train the value-only baseline

- Two pre-normalization Transformer blocks, width 64, four heads, feed-forward width 128, learned token/position embeddings, no dropout; exactly **74,659 parameters**. Keep the provided config unchanged.
- **300 updates**, batch size 128, AdamW lr `1e-3`, weight decay `1e-4`, gradient clipping at norm 1; model and data seed 3230. Every experiment starts from random initialization and uses the final update.
- Same raw training problems, same initialization and same update count for every format. Each problem's response is encoded with the submitted template.
- CPU inference/training with two PyTorch threads in the fixed grading environment. The course creates the fixed model and training loop. Attention and causal masks call your submitted functions, and the loss uses your validated inputs, targets and loss mask. Your functions therefore affect the learned model and its predictions.
- Greedy full-vocabulary decoding. There is no beam search, sampling, test-time repair or prompt-specific branch.



After completing Section 2.2, train the baseline response format:

```json
{"version": 1, "record": ["value"]}
```

Use the starting `submission/format.json` with this record and run:

```bash
python train_transformer.py --submission submission --result baseline_result.json
```

The command validates your functions, trains the fixed model using them, and evaluates 512 validation problems. Record its strict response accuracy before exploring other formats. If the implementation checks fail, correct your functions before proceeding.

The optional cell below explicitly selects the value-only baseline using a temporary format file, so it also works after you have edited `submission/format.json`. It does not change your chosen format. Training is disabled by default.

In [ ]:
RUN_BASELINE_TRAINING = False
if RUN_BASELINE_TRAINING:
    import tempfile
    with tempfile.TemporaryDirectory(prefix="a3_baseline_") as baseline_dir:
        baseline_format_path = Path(baseline_dir) / "format.json"
        baseline_format_path.write_text(json.dumps(BASELINE))
        subprocess.run([sys.executable, "train_transformer.py", "--submission", str(SUBMISSION_DIR),
                        "--format", str(baseline_format_path),
                        "--result", "baseline_result.json"], check=True)
    baseline_result = json.loads((ROOT / "baseline_result.json").read_text())
    print("Value-only baseline strict response accuracy:", baseline_result["metrics"]["strict_response_accuracy"])
else:
    print("Complete transformer.py, then enable this cell to train the value-only baseline.")

### 2.4 · Design your output format

After running the baseline, edit `submission/format.json` to explore a different response representation. The baseline format is:

```json
{"version": 1, "record": ["value"]}
```

`record` is repeated once per query, in query order. The course encoder adds one final EOS automatically. A record contains **1–5 fields and exactly one `value`**. Other fields may repeat. Only the following strings are allowed:

| Field | Token to generate for query i |
|---|---|
| `value` | The dictionary value associated with that query |
| `key` | That query's key |
| `index` | Its one-based query position, I1–I4 |
| `bar` | A fixed `|` marker |
| `dot` | A fixed `.` marker |

You choose the order and repetitions of these fields. No custom Python encoder/decoder, prompt changes, arbitrary literal strings, conditional rules or extra JSON fields are accepted. A record cannot omit or repeat `value`. JSON duplicate keys are rejected; the file must be at most 4096 bytes.

The maximum generation cap is `4 * len(record) + 1`, at most 21 new tokens. The longest complete prompt-plus-response is 44 tokens, within the model's 48-position capacity.

The parser reads **generated tokens only**. It checks the declared layout, field token types, index/marker values, record count and EOS, then extracts values. It cannot look up the dictionary, fill missing answers, reorder values, or repair errors. Every extra output field must be generated by the model.

The preview below demonstrates the format language; it is not a recommended high-performing answer.

In [ ]:
from assignment_lib.format_spec import validate_format, parse_answer, read_format
preview = {"version": 1, "record": ["index", "value", "dot"]}
validate_format(preview)
encoded = encode_answer([12, 5, 7], [22, 27, 24], preview)
print("Example format:", preview)
print("Encoded response:", format_tokens(encoded))
print("Extracted value IDs and valid flag:", parse_answer(encoded, preview))
print("Your current format:", read_format(SUBMISSION_DIR / "format.json"))

### 2.5 · Compare your format with the baseline

Run your chosen format and compare it with value-only using the same submitted functions and fixed training settings:

```bash
python train_transformer.py --submission submission --compare-baseline
```

The command trains the fixed model, evaluates 512 validation problems, compares the baseline on those same problems, and saves `format_result.json`. The `.format_cache/` directory holds local experimental weights/results; **do not submit it**. A cache entry is reused only when the entire submitted `transformer.py`, format, fixed training policy, course code and runtime versions match. Different student implementations do not share a result merely because they use the same format.

You may explore multiple formats using validation results. Do not select formats on the test partition. The grading server ignores your local weights, logs and scores: it loads your functions, validates your JSON, and trains from scratch or reuses a matching implementation-and-format result. Public self-check and formal grading use the same policy; formal evaluation uses 4096 unseen test-partition problems.

**Equal updates do not imply equal compute.** Longer records have more response targets and longer sequences. Inspect the reported input-token count, supervised-token count and runtime before attributing differences entirely to representation. These counts are diagnostics, not separate score components.

In [ ]:
RUN_FORMAT_EXPERIMENT = False
if RUN_FORMAT_EXPERIMENT:
    subprocess.run([sys.executable, "train_transformer.py", "--submission", str(SUBMISSION_DIR),
                    "--compare-baseline", "--result", "format_result.json"], check=True)
else:
    print("Complete transformer.py and choose format.json, then enable this comparison.")

In [ ]:
SHOW_FORMAT_RESULTS = False
if SHOW_FORMAT_RESULTS:
    result = json.loads((ROOT / "format_result.json").read_text())
    rows = [("your format", result)]
    if "baseline" in result: rows.append(("baseline", result["baseline"]))
    for name, run in rows:
        metrics = run["metrics"]
        print(name, run["format"]["record"])
        print("Strict response accuracy:", metrics["strict_response_accuracy"])
        print("Value-sequence / value-token accuracy:", metrics["value_sequence_accuracy"], metrics["value_token_accuracy"])
        print("By query count:", metrics["by_query_count"])
        print("Supervised tokens / time:", run["training"]["supervised_tokens"], run["training"]["train_seconds"])
else:
    print("Enable after an experiment to inspect accuracy and training cost.")

### 2.6 · Format and generation scoring — 25 points

A valid `format.json` earns **5 points**. The generation component earns up to **20 points** using strict response accuracy $a$ after the fixed 300-update training run:

$$
S_{\mathrm{format,acc}}=20\;\mathrm{clip}(a/0.95,0,1).
$$

A strict match requires every generated response token, including any auxiliary fields and EOS, to equal the response specified by your format. Wrong keys, misplaced markers, missing EOS or extra records fail that example. This prevents an incorrect auxiliary sequence from earning generation marks merely because some values match.

The report also includes value-sequence accuracy, value-token accuracy, valid-syntax rate and results by query count. The common value metrics extract the generated values in order; incorrect auxiliary key identities may leave these diagnostic metrics high while strict accuracy is lower. Invalid layout/index/marker tokens or missing EOS give zero value-token credit for that response; missing values count as wrong.

There is no required format or performance gap relative to baseline. Marks depend on the submitted design's absolute score, not other students' ranks. Your implementation must be trainable and causal: invalid context shapes, missing gradient paths, noncausal masks/future leakage, or incorrect target/loss alignment stop the run and earn zero generation points. Correct function-test marks and the 5 JSON-validity points remain available. Not every numerical test is an all-or-nothing gate: an incorrect returned diagnostic attention-weight tensor loses its function points but does not invalidate otherwise correct context calculations.

### 2.7 · One-page analysis — 5 points, marked by the TA

Submit a **one-page PDF named `analysis.pdf`**, placed in your `submission/` directory so it is included in your submission ZIP. Address these three questions:

1. What output format did you choose, and why?
2. Compared with the baseline that outputs only `value`, is your validation accuracy higher, lower, or the same? Report both strict response accuracies using the same implementation and fixed training settings. Run `python train_transformer.py --submission submission --compare-baseline` to obtain this comparison.
3. What might explain the result? Connect your explanation to your chosen format and observations. Distinguish a plausible explanation from something your experiment establishes; a lower or unchanged accuracy is also a valid outcome.

A small results table is sufficient; additional experiments are not required. **Submitting the PDF earns all 5 points.** These are submission-credit points: the TA does not grade the quality or correctness of the explanation, and no improvement over the baseline is required. If no PDF is submitted, this component earns 0. The automated evaluator does not inspect or score the PDF.

## Check and submit

Quick loading/shape/format validation, without format training:

```bash
python check_submission.py --submission example_submission --interfaces-only
python check_submission.py --submission submission --interfaces-only
```

Full public check:

```bash
python check_submission.py --submission submission --data-dir data
```

The checker trains or reuses your implementation-and-format experiment in a **separate process**, then runs function and CNN checks in another fresh worker. It uses CPU/two threads with independent default limits: `--format-timeout 300` for the format experiment (including cache waiting), and `--timeout 300` for correctness checks. The stages run sequentially, so together they may wait up to about 600 seconds, plus process startup/cleanup. A format timeout does not reduce the correctness budget. JSON results preserve per-component scores, errors and cache status. Public and official model evaluation share the model, data rules, format parser and scoring formula; evaluation split, random seed and sample count differ. Formal grading additionally enforces the required acknowledgements.

Without Fashion-MNIST data, CNN accuracy is SKIP (25 points); the remaining automated report covers 70 points. The full automated assessment is out of 95; the TA adds the 5 PDF submission points separately. An accuracy PASS means evaluation completed, not that it earned full marks. Exit code 0 means all executed checks passed. A timeout or crash never reuses an old report. Invalid student functions or exceptions during training give a FAIL for generation while preserving other component scores. If the training process times out or encounters an infrastructure failure, generation is ERROR with points pending; the report retains correctness scores and sets `needs_retry`. The partial score excludes the pending 20 points and is not a final grade. Re-run the same check to retry; completed cached training can be reused.

The supplied example has random CNN weights and intentionally incorrect Transformer functions. It passes interfaces and format validation, while failing several correctness checks. Its training precheck rejects the incorrect functions, so its generation component receives no points. Complete the functions before training your own model; the example is not a reference solution.

Package your final submission:

```bash
python pack_submission.py --submission submission --student-id YOUR_STUDENT_ID
```

Submit one `<student_id>_assignment3.zip`, containing **seven files at the ZIP root**:

```text
cnn.py
cnn_weights.pt
transformer.py
config.json
format.json
analysis.pdf
acknowledgement.json
```

`acknowledgement.json` must contain both `"cnn": "yes"` and `"transformer": "yes"`. The packer copies this file if present but deliberately does not check it; public checks also do not check it. Missing or invalid acknowledgements result in 0/100 during formal grading, regardless of code or PDF submission.

`cnn_training.json` may also be included. The packer includes `analysis.pdf` when present. Missing the PDF does not prevent code grading, but forfeits the 5 manual submission points. No Transformer weights, local result JSON, cache, dataset, environment, notebook or replacement helper library is submitted. The CNN checkpoint is at most 8 MB; expanded submission size is at most 12 MB.

Keep your working notes and experiments. The one-page PDF is the only required written analysis. Follow the course collaboration/AI-use policy, acknowledge permitted external code, and do not use test labels, pretrained weights, hard-coded answers, grading-process inspection or evaluator-specific behavior.